# UPR-MVS Server Debug Notebook

这份 notebook 面向服务器单张 A100 的逐步调试，不再走 `torchrun` / DDP，而是直接复用项目里的 `datasets/`、`models/`、`engine/`、`train.py`、`test.py`。

建议顺序：
1. 先修改“运行参数”单元里的路径、阶段和 checkpoint。
2. 先跑到“样本检查”和“单步前向”，确认路径、shape、显存、NaN 都正常。
3. 再执行“单步反向”和“阶段训练”。
4. 需要看源码时，直接用 `show_source(...)` 查看任意函数/类的实现。

这份 notebook 不会改写项目原始模块，适合在 Jupyter 里一格一格定位问题。

In [ ]:
import gc
import inspect
import json
import os
import random
import sys
import time
import traceback
from copy import deepcopy
from pathlib import Path

PROJECT_ROOT = Path("/scr/user/qinglong/projects/UPR-MVS")
SERVER_DTU_TRAIN_ROOT = Path("/scr/user/qinglong/dataset/DTU/dtu_training")
SERVER_DTU_TEST_ROOT = Path("/scr/user/qinglong/dataset/DTU/dtu_test")
SERVER_DA3_PRETRAINED = Path("/CHANGE_ME/DA3METRIC-LARGE.pth")

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("NCCL_DEBUG", "ERROR")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

import numpy as np
import torch
import yaml
from IPython.display import Code, Markdown, display
from torch.utils.data import DataLoader, Subset

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SERVER_DTU_TRAIN_ROOT = {SERVER_DTU_TRAIN_ROOT}")
print(f"SERVER_DTU_TEST_ROOT = {SERVER_DTU_TEST_ROOT}")
print(f"SERVER_DA3_PRETRAINED = {SERVER_DA3_PRETRAINED}")
print(f"Torch = {torch.__version__}")
print(f"CUDA available = {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_props = torch.cuda.get_device_properties(0)
    print(f"GPU = {torch.cuda.get_device_name(0)}")
    print(f"VRAM = {device_props.total_memory / 1024 ** 3:.1f} GB")


## 运行参数

先改这一格。当前 notebook 默认走 DA3 metric prior + point refinement。`stage_b` 通常需要把 `RESUME_CHECKPOINT` 指到 `stage_a/best.pth`。Jupyter 里默认关闭 DDP，方便单卡逐步调试。

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs/server_training.config"
STAGE_NAME = "stage_a"  # stage_a / stage_b
WORK_DIR = PROJECT_ROOT / "saved/notebook_single_a100"

SPLIT = "train"
VAL_SPLIT = "val"
DEBUG_SAMPLE_INDEX = 0

FAST_DEBUG = True
DEBUG_NUM_TRAIN_SAMPLES = 8 if FAST_DEBUG else None
DEBUG_NUM_VAL_SAMPLES = 4 if FAST_DEBUG else None
DEBUG_BATCH_SIZE = None      # None -> 使用当前 stage 的 batch size
DEBUG_EPOCHS = 1 if FAST_DEBUG else None
DEBUG_NUM_WORKERS = 0        # Jupyter 建议先用 0，稳定后再改大
DEBUG_MAX_EVAL_BATCHES = 2

RESUME_CHECKPOINT = None     # 例如: WORK_DIR / 'stage_a' / 'best.pth'
LOAD_TRAINING_STATE = False  # 只做阶段衔接时通常保持 False

DEPTH_PRIOR_OVERRIDE = SERVER_DA3_PRETRAINED
DATA_PATH_OVERRIDES = {
    "train_root": SERVER_DTU_TRAIN_ROOT,
    "val_root": SERVER_DTU_TRAIN_ROOT,
    "test_root": SERVER_DTU_TEST_ROOT,
    "test_gt_root": SERVER_DTU_TRAIN_ROOT,
}

AMP_DTYPE_OVERRIDE = "bf16"
ENABLE_TENSORBOARD = False
ENABLE_AUTOGRAD_ANOMALY = False

if DEPTH_PRIOR_OVERRIDE and not Path(DEPTH_PRIOR_OVERRIDE).exists():
    print(f"[WARNING] DA3 checkpoint not found yet: {DEPTH_PRIOR_OVERRIDE}")

if ENABLE_AUTOGRAD_ANOMALY:
    torch.autograd.set_detect_anomaly(True)

print(json.dumps(
    {
        "config": str(CONFIG_PATH),
        "stage": STAGE_NAME,
        "work_dir": str(WORK_DIR),
        "fast_debug": FAST_DEBUG,
        "resume": str(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else None,
        "amp_dtype": AMP_DTYPE_OVERRIDE,
        "num_workers": DEBUG_NUM_WORKERS,
        "train_root": str(DATA_PATH_OVERRIDES["train_root"]),
        "val_root": str(DATA_PATH_OVERRIDES["val_root"]),
        "test_root": str(DATA_PATH_OVERRIDES["test_root"]),
        "test_gt_root": str(DATA_PATH_OVERRIDES["test_gt_root"]),
        "depth_prior": str(DEPTH_PRIOR_OVERRIDE) if DEPTH_PRIOR_OVERRIDE else None,
    },
    indent=2,
    ensure_ascii=False,
))


## 导入项目模块与辅助函数

这一格会把项目原始模块挂进 notebook，同时补一些调试用的辅助函数，比如 shape 树、源码查看、hook 和单卡阶段构建。

In [ ]:
from datasets.dtu import build_dtu_dataset
from engine.checkpoint_io import load_checkpoint
from engine.ddp_utils import move_to_device, unwrap_model
from engine.trainer import (
    UPRMVSTrainer,
    autocast_context,
    build_grad_scaler,
    build_optimizer,
    build_scheduler,
    configure_trainable_modules,
)
from models.losses import UPRMVSLoss
from models.point.feature_lifting import project_world_to_view
from models.point.unproject import depth_to_world_points
from models.transformer.utils import scale_intrinsics
from train import (
    apply_stage_config,
    build_model,
    load_config as load_train_config,
    resolve_train_batch_size,
    update_config_for_stage,
)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def maybe_override_paths(config):
    for key, value in DATA_PATH_OVERRIDES.items():
        if value:
            config["data"][key] = str(Path(value))
    if DEPTH_PRIOR_OVERRIDE:
        config.setdefault("model", {}).setdefault("depth_anything3", {})["pretrained"] = str(Path(DEPTH_PRIOR_OVERRIDE))
    return config


def make_stage_config(stage_name):
    config = deepcopy(load_train_config(CONFIG_PATH))
    config = update_config_for_stage(config, stage_name)
    config = maybe_override_paths(config)
    config["train"]["use_ddp"] = False
    config["train"]["num_workers"] = int(DEBUG_NUM_WORKERS)
    if AMP_DTYPE_OVERRIDE:
        config["train"]["amp_dtype"] = AMP_DTYPE_OVERRIDE
    if DEBUG_EPOCHS is not None:
        config["train"]["epochs"] = int(DEBUG_EPOCHS)
    if DEBUG_BATCH_SIZE is not None:
        config["train"]["batch_size_per_gpu"] = int(DEBUG_BATCH_SIZE)
    if not ENABLE_TENSORBOARD:
        tensorboard_cfg = config["train"].setdefault("tensorboard", {})
        tensorboard_cfg["enable"] = False
    return config


def limit_dataset(dataset, limit):
    if limit is None or limit >= len(dataset):
        return dataset
    return Subset(dataset, list(range(limit)))


def make_loader(dataset, batch_size, shuffle):
    num_workers = int(DEBUG_NUM_WORKERS)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=shuffle,
        persistent_workers=num_workers > 0,
    )


def tensor_tree(obj, prefix=""):
    if torch.is_tensor(obj):
        return f"{prefix}{tuple(obj.shape)} | {obj.dtype} | {obj.device}"
    if isinstance(obj, dict):
        lines = []
        for key, value in obj.items():
            lines.append(f"{prefix}{key}:")
            lines.append(tensor_tree(value, prefix + "  "))
        return "\n".join(lines)
    if isinstance(obj, (list, tuple)):
        lines = []
        for idx, value in enumerate(obj):
            lines.append(f"{prefix}[{idx}]:")
            lines.append(tensor_tree(value, prefix + "  "))
        return "\n".join(lines)
    return f"{prefix}{type(obj).__name__}: {obj}"


def count_parameters(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable


def pretty_param_table(model):
    target = unwrap_model(model)
    rows = []
    for name in ["depth_prior", "backbone", "feature_lifter", "point_refiner", "densifier"]:
        if hasattr(target, name):
            total, trainable = count_parameters(getattr(target, name))
            rows.append((name, total, trainable))
    return rows


def report_cuda_memory():
    if not torch.cuda.is_available():
        return {"allocated_gb": 0.0, "reserved_gb": 0.0, "max_allocated_gb": 0.0}
    return {
        "allocated_gb": torch.cuda.memory_allocated() / 1024 ** 3,
        "reserved_gb": torch.cuda.memory_reserved() / 1024 ** 3,
        "max_allocated_gb": torch.cuda.max_memory_allocated() / 1024 ** 3,
    }


def reset_cuda_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def show_source(obj):
    display(Code(inspect.getsource(obj), language="python"))


def build_pixel_coords(height, width, device, dtype=torch.float32):
    ys, xs = torch.meshgrid(
        torch.arange(height, device=device, dtype=dtype),
        torch.arange(width, device=device, dtype=dtype),
        indexing="ij",
    )
    return torch.stack([xs, ys], dim=-1).view(1, height * width, 2)


def depth_roundtrip_stats(depth_map, intrinsics, extrinsics, mask=None, max_points=8192):
    if depth_map.ndim == 3:
        depth_map = depth_map.squeeze(0)
    if mask is None:
        mask = depth_map > 0
    elif mask.ndim == 3:
        mask = mask.squeeze(0) > 0.5
    else:
        mask = mask > 0.5

    height, width = depth_map.shape[-2:]
    pixel_coords = build_pixel_coords(height, width, depth_map.device, dtype=depth_map.dtype)
    depth_flat = depth_map.view(1, -1)
    mask_flat = mask.view(-1)
    valid_idx = torch.nonzero(mask_flat & torch.isfinite(depth_flat.view(-1)) & (depth_flat.view(-1) > 0), as_tuple=False).squeeze(-1)

    if valid_idx.numel() == 0:
        return {"num_valid": 0, "pixel_error_mean": float("nan"), "pixel_error_max": float("nan"), "depth_error_mean": float("nan"), "depth_error_max": float("nan")}

    if valid_idx.numel() > max_points:
        perm = torch.randperm(valid_idx.numel(), device=depth_map.device)[:max_points]
        valid_idx = valid_idx[perm]

    sampled_depth = depth_flat[:, valid_idx]
    sampled_pixels = pixel_coords[:, valid_idx]
    points_world, points_cam = depth_to_world_points(
        depth=sampled_depth,
        intrinsics=intrinsics.unsqueeze(0),
        extrinsics=extrinsics.unsqueeze(0),
        pixel_coords=sampled_pixels,
    )
    reproj_pixels, reproj_depth = project_world_to_view(
        points_world=points_world,
        intrinsics=intrinsics.unsqueeze(0),
        extrinsics=extrinsics.unsqueeze(0),
    )
    pixel_error = (reproj_pixels - sampled_pixels).norm(dim=-1)
    depth_error = (reproj_depth.unsqueeze(-1) - points_cam[..., 2:3]).abs().squeeze(-1)
    return {
        "num_valid": int(valid_idx.numel()),
        "pixel_error_mean": float(pixel_error.mean().detach().cpu()),
        "pixel_error_max": float(pixel_error.max().detach().cpu()),
        "depth_error_mean": float(depth_error.mean().detach().cpu()),
        "depth_error_max": float(depth_error.max().detach().cpu()),
    }


def summarize_depth_tensor(name, depth_tensor, mask_tensor=None):
    depth = depth_tensor.detach().float()
    if depth.ndim == 3:
        depth = depth.squeeze(0)
    valid_mask = torch.isfinite(depth) & (depth > 0)
    if mask_tensor is not None:
        mask = mask_tensor.detach().float()
        if mask.ndim == 3:
            mask = mask.squeeze(0)
        valid_mask = valid_mask & (mask > 0.5)
    valid_values = depth[valid_mask]
    if valid_values.numel() == 0:
        return {"name": name, "num_valid": 0}
    return {
        "name": name,
        "num_valid": int(valid_values.numel()),
        "min": float(valid_values.min().cpu()),
        "max": float(valid_values.max().cpu()),
        "mean": float(valid_values.mean().cpu()),
        "std": float(valid_values.std().cpu()) if valid_values.numel() > 1 else 0.0,
    }


def attach_top_level_shape_hooks(model, module_names=None):
    target = unwrap_model(model)
    module_names = module_names or ["depth_prior", "feature_lifter", "point_refiner", "densifier"]
    records = []
    handles = []

    def summarize(value):
        if torch.is_tensor(value):
            return {
                "shape": tuple(value.shape),
                "dtype": str(value.dtype),
                "device": str(value.device),
            }
        if isinstance(value, dict):
            return {k: summarize(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return [summarize(v) for v in value]
        return type(value).__name__

    def make_hook(name):
        def hook(_module, inputs, outputs):
            records.append({
                "module": name,
                "inputs": summarize(inputs),
                "outputs": summarize(outputs),
            })
        return hook

    for name in module_names:
        module = getattr(target, name, None)
        if module is not None:
            handles.append(module.register_forward_hook(make_hook(name)))
    return handles, records


def remove_hooks(handles):
    for handle in handles:
        handle.remove()


def build_stage_runtime(stage_name, resume_checkpoint=None, load_training_state=False):
    reset_cuda_memory()
    config = make_stage_config(stage_name)
    config_path = Path(CONFIG_PATH).resolve()
    work_dir = Path(WORK_DIR).resolve() / stage_name
    work_dir.mkdir(parents=True, exist_ok=True)

    set_seed(int(config["train"].get("seed", 42)))
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    train_dataset_full = build_dtu_dataset(
        config["data"],
        split=SPLIT,
        project_root=PROJECT_ROOT,
        config_dir=config_path.parent,
    )
    val_dataset_full = build_dtu_dataset(
        config["data"],
        split=VAL_SPLIT,
        project_root=PROJECT_ROOT,
        config_dir=config_path.parent,
    )
    train_dataset = limit_dataset(train_dataset_full, DEBUG_NUM_TRAIN_SAMPLES)
    val_dataset = limit_dataset(val_dataset_full, DEBUG_NUM_VAL_SAMPLES)

    batch_size = int(DEBUG_BATCH_SIZE or resolve_train_batch_size(config["train"]))
    train_loader = make_loader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = make_loader(val_dataset, batch_size=1, shuffle=False)

    model = build_model(config).to(device)
    configure_trainable_modules(
        model,
        str(config["train"].get("stage", "point_refine")).lower(),
        config.get("loss"),
    )
    apply_stage_config(model, config, stage_name, device)

    criterion = UPRMVSLoss(config["loss"]).to(device)
    optimizer = build_optimizer(model, config)
    scheduler = build_scheduler(optimizer, config)
    use_fp16_scaler = str(config["train"].get("amp_dtype", "bf16")).lower() == "fp16" and device.type == "cuda"
    scaler = build_grad_scaler(enabled=use_fp16_scaler)

    start_epoch = 0
    best_metric = float("inf")
    if resume_checkpoint:
        start_epoch, best_metric = load_checkpoint(
            checkpoint_path=resume_checkpoint,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            device=device,
            load_training_state=load_training_state,
        )

    trainer = UPRMVSTrainer(
        model=model,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=device,
        train_cfg=config["train"],
        work_dir=work_dir,
    )

    return {
        "config": config,
        "config_path": config_path,
        "device": device,
        "work_dir": work_dir,
        "train_dataset_full": train_dataset_full,
        "val_dataset_full": val_dataset_full,
        "train_dataset": train_dataset,
        "val_dataset": val_dataset,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "model": model,
        "criterion": criterion,
        "optimizer": optimizer,
        "scheduler": scheduler,
        "scaler": scaler,
        "trainer": trainer,
        "start_epoch": start_epoch,
        "best_metric": best_metric,
        "batch_size": batch_size,
    }


def evaluate_n_batches(model, criterion, loader, device, amp_dtype, max_batches=None):
    model.eval()
    metrics = []
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            if max_batches is not None and batch_idx >= max_batches:
                break
            batch = move_to_device(batch, device)
            with autocast_context(device, amp_dtype):
                outputs = model(batch)
                loss_dict = criterion(outputs, batch)
            metrics.append({k: float(v.detach().cpu()) for k, v in loss_dict.items()})
    if not metrics:
        return {}
    keys = sorted(metrics[0].keys())
    return {key: float(np.mean([item[key] for item in metrics])) for key in keys}


def run_all_stages_single_gpu(stages=("stage_a", "stage_b")):
    previous_best = None
    stage_results = {}
    for idx, stage_name in enumerate(stages):
        resume = RESUME_CHECKPOINT if idx == 0 else previous_best
        runtime = build_stage_runtime(
            stage_name=stage_name,
            resume_checkpoint=resume,
            load_training_state=False if idx > 0 else LOAD_TRAINING_STATE,
        )
        print(f"\\n===== {stage_name} =====")
        runtime["trainer"].fit(
            train_loader=runtime["train_loader"],
            val_loader=runtime["val_loader"],
            train_sampler=None,
            start_epoch=runtime["start_epoch"],
            max_epochs=int(runtime["config"]["train"]["epochs"]),
            best_metric=runtime["best_metric"],
        )
        previous_best = runtime["work_dir"] / "best.pth"
        stage_results[stage_name] = {
            "work_dir": runtime["work_dir"],
            "best_checkpoint": previous_best,
        }
        runtime["trainer"].close()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return stage_results


print("项目函数已导入，可以开始构建 notebook 运行时。")


## 构建当前阶段运行时

这一格会真正读取配置、建数据集、建 dataloader、建模型、建 loss / optimizer / scheduler / trainer。

In [ ]:
runtime = build_stage_runtime(
    stage_name=STAGE_NAME,
    resume_checkpoint=RESUME_CHECKPOINT,
    load_training_state=LOAD_TRAINING_STATE,
)

print(json.dumps(
    {
        "stage": STAGE_NAME,
        "train_stage": runtime["config"]["train"]["stage"],
        "work_dir": str(runtime["work_dir"]),
        "device": str(runtime["device"]),
        "batch_size": runtime["batch_size"],
        "train_dataset_len": len(runtime["train_dataset"]),
        "val_dataset_len": len(runtime["val_dataset"]),
        "amp_dtype": runtime["config"]["train"]["amp_dtype"],
        "num_workers": runtime["config"]["train"]["num_workers"],
        "resume_checkpoint": str(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else None,
    },
    indent=2,
    ensure_ascii=False,
))

for name, total, trainable in pretty_param_table(runtime["model"]):
    print(f"{name:>22}: total={total:,} | trainable={trainable:,}")

if STAGE_NAME in {"stage_b"} and RESUME_CHECKPOINT is None:
    print("提醒: 当前是 point/joint 阶段，但没有加载上一阶段 checkpoint，结果可能不稳定。")

print("CUDA memory:", report_cuda_memory())


## 样本检查

先确认 batch 键值、shape、路径和深度范围都对，再做前向。

In [ ]:
sample = runtime["train_dataset_full"][DEBUG_SAMPLE_INDEX]

print(f"sample_name  = {sample['sample_name']}")
print(f"scan_name    = {sample['scan_name']}")
print(f"view_ids     = {sample['view_ids'].tolist()}")
print(f"depth_range  = {sample['depth_range'].tolist()}")
print(f"has_depth_gt = {bool(sample['has_depth_gt'])}")
print(tensor_tree({k: v for k, v in sample.items() if k not in {'scan_name', 'sample_name'}}))

if plt is not None:
    ref_img = sample["imgs"][0].permute(1, 2, 0).numpy().clip(0.0, 1.0)
    depth_gt = sample["depth_gt"][0].numpy()
    mask = sample["mask"][0].numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].imshow(ref_img)
    axes[0].set_title("Reference image")
    axes[0].axis("off")
    axes[1].imshow(depth_gt, cmap="viridis")
    axes[1].set_title("Depth GT")
    axes[1].axis("off")
    axes[2].imshow(mask, cmap="gray")
    axes[2].set_title("Mask")
    axes[2].axis("off")
    plt.tight_layout()
else:
    print("matplotlib 不可用，跳过可视化。")


## 源码查看

把 `SOURCE_TARGET` 换成任何你想看的函数或类，例如 `build_dtu_dataset`、`UPRMVSTrainer.train_one_epoch`、`UPRMVSLoss.forward`。

In [ ]:
SOURCE_TARGET = unwrap_model(runtime["model"]).forward
show_source(SOURCE_TARGET)

# 其他常用对象:
# show_source(build_dtu_dataset)
# show_source(UPRMVSTrainer.train_one_epoch)
# show_source(UPRMVSLoss.forward)


## 取一个 batch 并送到设备

这一格主要用来确认 dataloader、pin memory、device 搬运是否正常。

In [ ]:
batch = next(iter(runtime["train_loader"]))
batch_gpu = move_to_device(batch, runtime["device"])

print("Batch on device:")
print(tensor_tree({k: v for k, v in batch_gpu.items() if k not in {"scan_name", "sample_name"}}))
print("CUDA memory before forward:", report_cuda_memory())


## GT 几何 Round-Trip 检查

先只检查数据和几何，不碰网络。这里如果 pixel/depth round-trip 本身就不接近 0，后面的 coarse 和 point 全都会被带偏。

In [ ]:
ref_intrinsics_full = batch_gpu["intrinsics"][0, 0]
ref_extrinsics = batch_gpu["extrinsics"][0, 0]
gt_depth = batch_gpu["depth_gt"][0]
gt_mask = batch_gpu["mask"][0]

gt_depth_stats = summarize_depth_tensor("depth_gt", gt_depth, gt_mask)
gt_roundtrip = depth_roundtrip_stats(gt_depth, ref_intrinsics_full, ref_extrinsics, gt_mask)

print(json.dumps(
    {
        "depth_range": batch_gpu["depth_range"][0].detach().cpu().tolist(),
        "depth_stats": gt_depth_stats,
        "roundtrip": gt_roundtrip,
    },
    indent=2,
    ensure_ascii=False,
))


## 单步前向

这里会顺便给 depth prior / point 等顶层模块挂 hook，便于你直接看到模块输入输出结构。

注意：`backbone` 在模型里是通过 `forward_multiview(...)` 直接调用的，不是 `backbone(...)`，所以普通 forward hook 默认抓不到它；如果你要查 backbone 输出，更适合直接看源码或单独调用 `_encode_multiview_features(...)`。

In [ ]:
runtime["model"].eval()
handles, hook_records = attach_top_level_shape_hooks(runtime["model"])

with torch.no_grad():
    with autocast_context(runtime["device"], runtime["config"]["train"]["amp_dtype"]):
        outputs = runtime["model"](batch_gpu)

remove_hooks(handles)

print(f"output keys ({len(outputs)}):", sorted(outputs.keys()))
shape_summary = {k: list(v.shape) for k, v in outputs.items() if torch.is_tensor(v)}
print(json.dumps(shape_summary, indent=2, ensure_ascii=False))
print("CUDA memory after forward:", report_cuda_memory())

print("Top-level hook records:")
if not hook_records:
    print("没有捕获到 hook 记录；这通常说明对应模块不是通过 module(...) 调用的。")
for record in hook_records:
    print(f"- {record['module']}")
    print(json.dumps(record["outputs"], indent=2, ensure_ascii=False, default=str))


## Coarse Depth 几何与尺度诊断

这一格专门用来查 `depth_abs_error` 很大时到底是哪里出问题：深度范围不对、coarse 输出整体偏移、还是几何投影链条不一致。

In [ ]:
coarse_depth = outputs["coarse_depth"][0].detach()
coarse_h, coarse_w = coarse_depth.shape[-2:]
img_h, img_w = batch_gpu["imgs"].shape[-2:]

ref_intrinsics_coarse = scale_intrinsics(
    batch_gpu["intrinsics"][0, 0].unsqueeze(0),
    scale_x=coarse_w / float(img_w),
    scale_y=coarse_h / float(img_h),
).squeeze(0)

gt_depth_small = torch.nn.functional.interpolate(
    batch_gpu["depth_gt"][0].unsqueeze(0),
    size=(coarse_h, coarse_w),
    mode="bilinear",
    align_corners=False,
).squeeze(0)
gt_mask_small = torch.nn.functional.interpolate(
    batch_gpu["mask"][0].float().unsqueeze(0),
    size=(coarse_h, coarse_w),
    mode="nearest",
).squeeze(0) > 0.5

coarse_stats = summarize_depth_tensor("coarse_depth", coarse_depth, gt_mask_small)
gt_small_stats = summarize_depth_tensor("depth_gt_small", gt_depth_small, gt_mask_small)
coarse_roundtrip = depth_roundtrip_stats(coarse_depth, ref_intrinsics_coarse, ref_extrinsics, gt_mask_small)

valid_mask = gt_mask_small & torch.isfinite(gt_depth_small) & (gt_depth_small > 0)
if valid_mask.any():
    abs_error = (coarse_depth - gt_depth_small).abs()
    coarse_error_summary = {
        "depth_abs_error_mean": float(abs_error[valid_mask].mean().detach().cpu()),
        "depth_abs_error_median": float(abs_error[valid_mask].median().detach().cpu()),
        "depth_abs_error_max": float(abs_error[valid_mask].max().detach().cpu()),
        "valid_pixels": int(valid_mask.sum().detach().cpu()),
    }
else:
    coarse_error_summary = {"valid_pixels": 0}

if "depth_values" in outputs:
    depth_values = outputs["depth_values"][0].detach().float().cpu()
    coarse_bins_summary = {
        "num_bins": int(depth_values.numel()),
        "depth_bin_min": float(depth_values.min()),
        "depth_bin_max": float(depth_values.max()),
    }
else:
    coarse_bins_summary = {}

print(json.dumps(
    {
        "depth_range": batch_gpu["depth_range"][0].detach().cpu().tolist(),
        "coarse_stats": coarse_stats,
        "gt_small_stats": gt_small_stats,
        "coarse_roundtrip": coarse_roundtrip,
        "coarse_error": coarse_error_summary,
        "depth_bins": coarse_bins_summary,
    },
    indent=2,
    ensure_ascii=False,
))


## Loss 检查与预测可视化

先确认 `loss_total`、`loss_coarse`、point 相关损失是否正常，再决定要不要进入反向传播。

In [ ]:
with autocast_context(runtime["device"], runtime["config"]["train"]["amp_dtype"]):
    loss_dict = runtime["criterion"](outputs, batch_gpu)

loss_summary = {k: float(v.detach().cpu()) for k, v in loss_dict.items()}
print(json.dumps(loss_summary, indent=2, ensure_ascii=False))

if plt is not None and "coarse_depth" in outputs:
    coarse = outputs["coarse_depth"][0, 0].detach().float().cpu().numpy()
    depth_gt = batch["depth_gt"][0, 0].numpy()
    has_score = "point_selection_score" in outputs
    fig, axes = plt.subplots(1, 3 if has_score else 2, figsize=(15 if has_score else 10, 4))
    axes = np.atleast_1d(axes)
    axes[0].imshow(coarse, cmap="viridis")
    axes[0].set_title("DA3 / prior depth")
    axes[0].axis("off")
    axes[1].imshow(depth_gt, cmap="viridis")
    axes[1].set_title("GT depth")
    axes[1].axis("off")
    if has_score:
        score_map = outputs["point_selection_score"][0, 0].detach().float().cpu().numpy()
        axes[2].imshow(score_map, cmap="magma")
        axes[2].set_title("Point selection score")
        axes[2].axis("off")
    plt.tight_layout()

if "point_final_mask" in outputs:
    final_points = int(outputs["point_final_mask"][0].sum().item())
    print(f"final point count = {final_points}")


## 单步反向与优化器更新

这一格会重新取一个 batch，跑完整的 forward + backward + optimizer step。适合查梯度爆炸、NaN、OOM、requires_grad 配置是否正确。

In [ ]:
runtime["model"].train()
runtime["optimizer"].zero_grad(set_to_none=True)

train_batch = next(iter(runtime["train_loader"]))
train_batch = move_to_device(train_batch, runtime["device"])

with autocast_context(runtime["device"], runtime["config"]["train"]["amp_dtype"]):
    train_outputs = runtime["model"](train_batch)
    train_loss_dict = runtime["criterion"](train_outputs, train_batch)
    total_loss = train_loss_dict["loss_total"]

print("loss_total =", float(total_loss.detach().cpu()))

if not torch.isfinite(total_loss):
    raise RuntimeError(f"loss_total 非有限值: {total_loss}")

grad_clip = float(runtime["config"]["train"].get("grad_clip", 0.0))
if runtime["scaler"].is_enabled():
    runtime["scaler"].scale(total_loss).backward()
    if grad_clip > 0:
        runtime["scaler"].unscale_(runtime["optimizer"])
        torch.nn.utils.clip_grad_norm_(runtime["model"].parameters(), max_norm=grad_clip)
    runtime["scaler"].step(runtime["optimizer"])
    runtime["scaler"].update()
else:
    total_loss.backward()
    if grad_clip > 0:
        torch.nn.utils.clip_grad_norm_(runtime["model"].parameters(), max_norm=grad_clip)
    runtime["optimizer"].step()

runtime["optimizer"].zero_grad(set_to_none=True)

train_loss_summary = {k: float(v.detach().cpu()) for k, v in train_loss_dict.items()}
print(json.dumps(train_loss_summary, indent=2, ensure_ascii=False))
print("lr =", runtime["optimizer"].param_groups[0]["lr"])
print("CUDA memory after backward:", report_cuda_memory())


## 小规模验证

这里不会写 checkpoint，只会在有限个 batch 上快速看 loss 指标是否合理。

In [ ]:
val_metrics = evaluate_n_batches(
    model=runtime["model"],
    criterion=runtime["criterion"],
    loader=runtime["val_loader"],
    device=runtime["device"],
    amp_dtype=runtime["config"]["train"]["amp_dtype"],
    max_batches=DEBUG_MAX_EVAL_BATCHES,
)
print(json.dumps(val_metrics, indent=2, ensure_ascii=False))


## 运行当前阶段训练

确认前面的单步检查都没问题后，再执行这一格。它会真实写入 `WORK_DIR / STAGE_NAME` 下的 checkpoint 与日志。

In [ ]:
runtime["trainer"].fit(
    train_loader=runtime["train_loader"],
    val_loader=runtime["val_loader"],
    train_sampler=None,
    start_epoch=runtime["start_epoch"],
    max_epochs=int(runtime["config"]["train"]["epochs"]),
    best_metric=runtime["best_metric"],
)
print(f"训练完成，输出目录: {runtime['work_dir']}")


## 正式训练建议

notebook 适合逐格调试数据、几何、coarse depth、point branch 和 loss。真正整跑时，建议回到训练脚本，使用新的单次运行 curriculum 模式，而不是 notebook 里的独立阶段 runtime。

In [ ]:
# 调试阶段时可以继续手动调用单阶段 runtime:
# runtime = build_stage_runtime('stage_a')
#
# 正式训练建议在终端执行:
# torchrun --nproc_per_node=4 train.py \
#   --config configs/server_training.config \
#   --work_dir saved/server_multi_gpu \
#   --stage curriculum \
#   --launcher pytorch
